
# Notebook 4: Feature Engineering

## Project: ThreatTrace — CICIDS2017 Intrusion Detection

### Objective
Prepare the cleaned CICIDS2017 dataset for subsequent machine learning experiments by examining its features and identifying opportunities for feature engineering and selection.

### Input Dataset
`data/processed/cicids2017_cleaned.csv`

### Planned Activities
1. Load and validate the cleaned dataset.
2. Inspect feature names and data types.
3. Separate input features from the target label.
4. Identify constant and duplicate features.
5. Examine feature distributions and relationships.
6. Decide which feature transformations or removals are justified.
7. Save the prepared dataset and document the findings.

### Scope
This notebook focuses on feature analysis and preparation. Class balancing, model training, and model evaluation will be handled in later notebooks.


In [1]:

# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

from pathlib import Path

import pandas as pd
import numpy as np

# Display settings
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 120)

print("Libraries imported successfully.")


Libraries imported successfully.


In [2]:

# ============================================================
# 2. LOAD CLEANED DATASET
# ============================================================

PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "data/processed/cicids2017_cleaned.csv").exists():
    PROJECT_ROOT = Path("D:/ThreatTrace")

DATA_PATH = PROJECT_ROOT / "data/processed/cicids2017_cleaned.csv"

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found: {DATA_PATH}\n"
        "Check the project directory and Notebook 3 output."
    )

df = pd.read_csv(DATA_PATH, low_memory=False)

print("Dataset loaded successfully.")
print(f"Dataset shape: {df.shape}")
print(f"Number of rows: {df.shape[0]:,}")
print(f"Number of columns: {df.shape[1]}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / (1024**3):.2f} GB")

display(df.head())


Dataset loaded successfully.
Dataset shape: (2520798, 79)
Number of rows: 2,520,798
Number of columns: 79
Memory usage: 1.59 GB


,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,Bwd Packet Length Max,Bwd Packet Length Min,Bwd Packet Length Mean,Bwd Packet Length Std,Flow Bytes/s,Flow Packets/s,Flow IAT Mean,Flow IAT Std,Flow IAT Max,Flow IAT Min,Fwd IAT Total,Fwd IAT Mean,Fwd IAT Std,Fwd IAT Max,Fwd IAT Min,Bwd IAT Total,Bwd IAT Mean,Bwd IAT Std,Bwd IAT Max,Bwd IAT Min,Fwd PSH Flags,Bwd PSH Flags,Fwd URG Flags,Bwd URG Flags,Fwd Header Length,Bwd Header Length,Fwd Packets/s,Bwd Packets/s,Min Packet Length,Max Packet Length,Packet Length Mean,Packet Length Std,Packet Length Variance,FIN Flag Count,SYN Flag Count,RST Flag Count,PSH Flag Count,ACK Flag Count,URG Flag Count,CWE Flag Count,ECE Flag Count,Down/Up Ratio,Average Packet Size,Avg Fwd Segment Size,Avg Bwd Segment Size,Fwd Header Length.1,Fwd Avg Bytes/Bulk,Fwd Avg Packets/Bulk,Fwd Avg Bulk Rate,Bwd Avg Bytes/Bulk,Bwd Avg Packets/Bulk,Bwd Avg Bulk Rate,Subflow Fwd Packets,Subflow Fwd Bytes,Subflow Bwd Packets,Subflow Bwd Bytes,Init_Win_bytes_forward,Init_Win_bytes_backward,act_data_pkt_fwd,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,54865,3,2,0,12,0,6,6,6.0,0.0,0,0,0.0,0.0,4.000000e+06,666666.66670,3.0,0.0,3,3,3,3.0,0.0,3,3,0,0.0,0.0,0,0,0,0,0,0,40,0,666666.666700,0.000000,6,6,6.0,0.0,0.0,0,0,0,0,1,0,0,0,0,9.0,6.0,0.0,40,0,0,0,0,0,0,2,12,0,0,33,-1,1,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
1,55054,109,1,1,6,6,6,6,6.0,0.0,6,6,6.0,0.0,1.100917e+05,18348.62385,109.0,0.0,109,109,0,0.0,0.0,0,0,0,0.0,0.0,0,0,0,0,0,0,20,20,9174.311927,9174.311927,6,6,6.0,0.0,0.0,0,0,0,0,1,1,0,0,1,9.0,6.0,6.0,20,0,0,0,0,0,0,1,6,1,6,29,256,0,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2,55055,52,1,1,6,6,6,6,6.0,0.0,6,6,6.0,0.0,2.307692e+05,38461.53846,52.0,0.0,52,52,0,0.0,0.0,0,0,0,0.0,0.0,0,0,0,0,0,0,20,20,19230.769230,19230.769230,6,6,6.0,0.0,0.0,0,0,0,0,1,1,0,0,1,9.0,6.0,6.0,20,0,0,0,0,0,0,1,6,1,6,29,256,0,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
3,46236,34,1,1,6,6,6,6,6.0,0.0,6,6,6.0,0.0,3.529412e+05,58823.52941,34.0,0.0,34,34,0,0.0,0.0,0,0,0,0.0,0.0,0,0,0,0,0,0,20,20,29411.764710,29411.764710,6,6,6.0,0.0,0.0,0,0,0,0,1,1,0,0,1,9.0,6.0,6.0,20,0,0,0,0,0,0,1,6,1,6,31,329,0,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
4,54863,3,2,0,12,0,6,6,6.0,0.0,0,0,0.0,0.0,4.000000e+06,666666.66670,3.0,0.0,3,3,3,3.0,0.0,3,3,0,0.0,0.0,0,0,0,0,0,0,40,0,666666.666700,0.000000,6,6,6.0,0.0,0.0,0,0,0,0,1,0,0,0,0,9.0,6.0,0.0,40,0,0,0,0,0,0,2,12,0,0,32,-1,1,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN


In [3]:

# ============================================================
# 3. DATASET VALIDATION
# ============================================================

print("Dataset shape:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes.value_counts().rename_axis("Data Type").to_frame("Count"))

print("\nMissing values:", int(df.isna().sum().sum()))

numeric_df = df.select_dtypes(include=[np.number])

print(
    "Infinite values:",
    int(np.isinf(numeric_df.to_numpy()).sum())
)

print("Duplicate rows:", int(df.duplicated().sum()))

if "Label" not in df.columns:
    raise ValueError(
        "Target column 'Label' was not found. "
        "Inspect the column names before continuing."
    )

print("\nTarget label distribution:")
display(
    df["Label"]
    .value_counts(dropna=False)
    .rename_axis("Label")
    .to_frame("Number of Records")
)


Dataset shape: (2520798, 79)

Column names:
['Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Min Packet Length', 'Max Packet Length', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'PSH Flag Count', 'ACK Flag Count', 'URG Fl

,Count
Data Type,
int64,54
float64,24
str,1



Missing values: 0
Infinite values: 0
Duplicate rows: 0

Target label distribution:


,Number of Records
Label,
BENIGN,2095057
DoS Hulk,172846
DDoS,128014
PortScan,90694
DoS GoldenEye,10286
FTP-Patator,5931
DoS slowloris,5385
DoS Slowhttptest,5228
SSH-Patator,3219



## 4. Feature Quality Analysis

In this section, we examine the input features to identify:
- Constant features with no variation.
- Duplicate feature columns.
- Numerical and non-numerical columns.
- Features that may require further preprocessing.

The target column `Label` is excluded from feature-quality checks.


In [5]:

# ============================================================
# 4. FEATURE QUALITY ANALYSIS
# ============================================================

# Separate target from input features
target_col = "Label"

feature_cols = [col for col in df.columns if col != target_col]

print("Total columns:", df.shape[1])
print("Target column:", target_col)
print("Number of input features:", len(feature_cols))

# Separate numerical and non-numerical features
numeric_features = df[feature_cols].select_dtypes(
    include=[np.number]
).columns.tolist()

non_numeric_features = df[feature_cols].select_dtypes(
    exclude=[np.number]
).columns.tolist()

print("\nNumerical features:", len(numeric_features))
print("Non-numerical features:", len(non_numeric_features))

print("\nNon-numerical feature names:")
print(non_numeric_features)

# Identify constant features
constant_features = [
    col for col in feature_cols
    if df[col].nunique(dropna=False) <= 1
]

print("\nConstant features:", len(constant_features))
print(constant_features)

Total columns: 79
Target column: Label
Number of input features: 78

Numerical features: 78
Non-numerical features: 0

Non-numerical feature names:
[]

Constant features: 8
['Bwd PSH Flags', 'Bwd URG Flags', 'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate']



## 6. Removal of Constant Features

The feature-quality analysis identified eight constant features:

1. Bwd PSH Flags
2. Bwd URG Flags
3. Fwd Avg Bytes/Bulk
4. Fwd Avg Packets/Bulk
5. Fwd Avg Bulk Rate
6. Bwd Avg Bytes/Bulk
7. Bwd Avg Packets/Bulk
8. Bwd Avg Bulk Rate

Constant features contain only one unique value across the dataset. They provide no discriminative information and can be removed to reduce the input dimensionality.

The target column `Label` is retained because it contains the classification categories required for intrusion detection.


In [6]:

# ============================================================
# 6. REMOVE CONSTANT FEATURES
# ============================================================

# Store the original feature count
original_feature_count = len(feature_cols)

# Remove constant features from the input data
df_features = df.drop(columns=constant_features).copy()

# Keep the target column unchanged
assert "Label" in df_features.columns
assert df_features["Label"].equals(df["Label"])

# Calculate the updated feature count
remaining_features = [
    col for col in df_features.columns if col != "Label"
]

print("Original input features:", original_feature_count)
print("Constant features removed:", len(constant_features))
print("Remaining input features:", len(remaining_features))
print("Total columns after removal:", df_features.shape[1])

print("\nRemoved features:")
print(constant_features)

print("\nUpdated dataset shape:", df_features.shape)


Original input features: 78
Constant features removed: 8
Remaining input features: 70
Total columns after removal: 71

Removed features:
['Bwd PSH Flags', 'Bwd URG Flags', 'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate']

Updated dataset shape: (2520798, 71)


In [7]:

# ============================================================
# 7. VERIFY REMAINING FEATURES
# ============================================================

remaining_constant_features = [
    col for col in remaining_features
    if df_features[col].nunique(dropna=False) <= 1
]

print("Remaining constant features:", len(remaining_constant_features))
print(remaining_constant_features)

print("\nMissing values:", int(df_features.isna().sum().sum()))

numeric_remaining = df_features[remaining_features].select_dtypes(
    include=[np.number]
)

print(
    "Infinite values in input features:",
    int(np.isinf(numeric_remaining.to_numpy()).sum())
)

print("\nTarget class count:", df_features["Label"].nunique())
print("Target distribution:")
display(
    df_features["Label"]
    .value_counts()
    .rename_axis("Label")
    .to_frame("Records")
)


Remaining constant features: 0
[]

Missing values: 0
Infinite values in input features: 0

Target class count: 15
Target distribution:


,Records
Label,
BENIGN,2095057
DoS Hulk,172846
DDoS,128014
PortScan,90694
DoS GoldenEye,10286
FTP-Patator,5931
DoS slowloris,5385
DoS Slowhttptest,5228
SSH-Patator,3219



## 8. Feature Correlation Analysis

This section examines relationships between numerical input features using Pearson correlation.

Highly correlated features may contain overlapping information. We will identify pairs with strong absolute correlation and inspect them before deciding whether any further feature reduction is justified.

The target column `Label` is excluded from the correlation calculation. No features will be removed based on correlation alone in this step.


In [8]:

# ============================================================
# 8. FEATURE CORRELATION ANALYSIS
# ============================================================

# Select numerical input features only
X_numeric = df_features.drop(columns=["Label"]).select_dtypes(
    include=[np.number]
)

print("Number of numerical input features:", X_numeric.shape[1])

# Calculate the correlation matrix
correlation_matrix = X_numeric.corr()

print("Correlation matrix shape:", correlation_matrix.shape)

# Extract unique feature pairs with absolute correlation >= 0.95
upper_triangle = correlation_matrix.where(
    np.triu(np.ones(correlation_matrix.shape), k=1).astype(bool)
)

high_correlation_pairs = (
    upper_triangle.stack()
    .rename("correlation")
    .reset_index()
)

high_correlation_pairs.columns = [
    "Feature 1", "Feature 2", "Correlation"
]

high_correlation_pairs = high_correlation_pairs[
    high_correlation_pairs["Correlation"].abs() >= 0.95
].sort_values(
    by="Correlation",
    key=lambda values: values.abs(),
    ascending=False
)

print(
    "\nNumber of feature pairs with absolute correlation >= 0.95:",
    len(high_correlation_pairs)
)

display(high_correlation_pairs.head(30))


Number of numerical input features: 70
Correlation matrix shape: (70, 70)

Number of feature pairs with absolute correlation >= 0.95: 39


,Feature 1,Feature 2,Correlation
194,Total Fwd Packets,Subflow Fwd Packets,1.000000
2142,Fwd PSH Flags,SYN Flag Count,1.000000
2293,Fwd Header Length,Fwd Header Length.1,1.000000
2217,Fwd URG Flags,CWE Flag Count,1.000000
892,Bwd Packet Length Mean,Avg Bwd Segment Size,1.000000
611,Fwd Packet Length Mean,Avg Fwd Segment Size,1.000000
266,Total Backward Packets,Subflow Bwd Packets,1.000000
407,Total Length of Bwd Packets,Subflow Bwd Bytes,1.000000
335,Total Length of Fwd Packets,Subflow Fwd Bytes,0.999999
3836,Subflow Fwd Packets,Subflow Bwd Packets,0.999070



## 10. Interpretation of Feature Correlation

The correlation analysis examined 70 numerical input features and identified 39 feature pairs with an absolute Pearson correlation of at least 0.95.

### Key Findings

- Several features are perfectly or almost perfectly correlated.
- Some features represent equivalent or closely related network traffic measurements.
- Strong correlation suggests possible redundancy, but it does not prove that a feature is unnecessary for classification.
- Correlation does not establish causation.
- Features will not be removed solely because their correlation exceeds the threshold.

### Decision

The eight constant features have been removed because they contain no variation.

The remaining 70 input features will be preserved at this stage. Further feature selection should be evaluated using training data and appropriate machine learning experiments to avoid discarding useful intrusion-detection information.

Class imbalance remains significant, particularly for the rare attack classes. It will be addressed separately in a later stage.


In [9]:

# ============================================================
# 11. SAVE FEATURE-ENGINEERED DATASET
# ============================================================

OUTPUT_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = OUTPUT_DIR / "cicids2017_feature_engineered.csv"

print("Output file:", OUTPUT_PATH)
print("Dataset shape:", df_features.shape)

# Save without the pandas index
df_features.to_csv(OUTPUT_PATH, index=False)

print("\nDataset saved successfully.")
print(f"File size: {OUTPUT_PATH.stat().st_size / (1024**3):.2f} GB")


Output file: D:\ThreatTrace\data\processed\cicids2017_feature_engineered.csv
Dataset shape: (2520798, 71)

Dataset saved successfully.
File size: 0.78 GB


In [10]:

# ============================================================
# 12. VERIFY SAVED DATASET
# ============================================================

assert OUTPUT_PATH.exists(), "Output file was not created."
assert OUTPUT_PATH.stat().st_size > 0, "Output file is empty."

print("Output exists:", OUTPUT_PATH.exists())
print("Output size:", f"{OUTPUT_PATH.stat().st_size / (1024**3):.2f} GB")
print("Expected rows:", len(df_features))
print("Expected columns:", len(df_features.columns))
print("Target column preserved:", "Label" in df_features.columns)

print("\nFeatures removed:", len(constant_features))
print("Remaining input features:", len(remaining_features))
print("Target classes:", df_features["Label"].nunique())

print("\nFeature engineering verification completed.")

Output exists: True
Output size: 0.78 GB
Expected rows: 2520798
Expected columns: 71
Target column preserved: True

Features removed: 8
Remaining input features: 70
Target classes: 15

Feature engineering verification completed.



## 13. Final Conclusion

In this notebook, feature engineering was performed on the cleaned CICIDS2017 intrusion-detection dataset.

### Work Completed

1. Loaded and validated the cleaned dataset.
2. Examined the 78 numerical input features.
3. Identified eight constant features and removed them.
4. Verified that no constant features remained.
5. Confirmed that the dataset contained no missing or infinite values.
6. Analyzed Pearson correlations and identified 39 highly correlated feature pairs using an absolute correlation threshold of 0.95.
7. Preserved all 15 target classes and the `Label` column.
8. Saved the feature-engineered dataset separately from the original cleaned dataset.

### Final Dataset

- Total records: 2,520,798
- Input features: 70
- Target column: `Label`
- Total columns: 71
- Target classes: 15
- Output: `data/processed/cicids2017_feature_engineered.csv`

### Limitations and Next Steps

The correlation analysis identified potentially redundant features, but no additional features were removed based on correlation alone. The dataset remains imbalanced, and no scaling, resampling, or model training was performed.

The next stage should prepare a reproducible training and testing workflow, apply appropriate preprocessing without data leakage, and evaluate intrusion-detection models.
